In [ ]:
import xarray as xr
import numpy as np
import json
import pandas as pd
import os
import glob
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from global_land_mask import globe 
import warnings

# Matikan warning agar output bersih
warnings.filterwarnings("ignore")

def process_swh_wave_rose_final():
    print("GENERATOR V5: SWH SPATIAL + WAVE ROSES (DYNAMIC)")

    # ==========================================
    # 0. KONFIGURASI VISUAL (SINKRON DENGAN ANGIN)
    # ==========================================
    CMAP_MAP = "YlGnBu"     
    ARROW_COLOR = "#000000"  # Vektor Hitam per request
    COASTLINE_COLOR = "#000000" 
    TEXT_COLOR_ROSE = "white" # Wave Rose menggunakan tema putih kontras
    TEXT_COLOR_CB = "#e0e0e0" # Colorbar menggunakan tema abu-abu terang
    
    DPI_MAP = 150
    DPI_ROSE = 100 
    
    # 1. KONFIGURASI DATA
    start_year = 1979
    end_year = 2024
    data_folder = '.' 
    
    # Folder Output
    base_dir = "frames_swh_final"
    dirs = {
        "map_ts": f"{base_dir}/map/ts",
        "map_clim": f"{base_dir}/map/clim",
        "map_seas": f"{base_dir}/map/seas",
        "rose_area_ts": f"{base_dir}/rose/area/ts",
        "rose_area_clim": f"{base_dir}/rose/area/clim",
        "rose_area_seas": f"{base_dir}/rose/area/seas",
        "rose_grid_ts": f"{base_dir}/rose/grid/ts",
        "rose_grid_clim": f"{base_dir}/rose/grid/clim",
        "rose_grid_seas": f"{base_dir}/rose/grid/seas"
    }
    for d in dirs.values():
        os.makedirs(d, exist_ok=True)

    monthly_ds_merged = []

    # ==========================================
    # 2. LOAD DATA (SWH & MWD)
    # ==========================================
    print(f"Membaca Data SWH & MWD {start_year}-{end_year}...")

    for year in range(start_year, end_year + 1):
        swh_files = glob.glob(os.path.join(data_folder, f"*significant_height*{year}*.nc"))
        mwd_files = glob.glob(os.path.join(data_folder, f"*mean_wave_direction*{year}*.nc"))
        
        if not swh_files or not mwd_files: continue
        
        try:
            ds_swh = xr.open_dataset(swh_files[0])
            ds_mwd = xr.open_dataset(mwd_files[0])
            
            if 'valid_time' in ds_swh.coords: ds_swh = ds_swh.rename({'valid_time': 'time'})
            if 'valid_time' in ds_mwd.coords: ds_mwd = ds_mwd.rename({'valid_time': 'time'})
            
            var_swh = next((v for v in ['swh', 'significant_height'] if v in ds_swh), None)
            var_mwd = next((v for v in ['mwd', 'mean_wave_direction'] if v in ds_mwd), None)
            
            if var_swh and var_mwd:
                u_comp = np.sin(np.deg2rad(ds_mwd[var_mwd]))
                v_comp = np.cos(np.deg2rad(ds_mwd[var_mwd]))
                
                ds_m = xr.Dataset({
                    'swh': ds_swh[var_swh].resample(time='1M').mean(),
                    'u_wave': u_comp.resample(time='1M').mean(),
                    'v_wave': v_comp.resample(time='1M').mean()
                })
                monthly_ds_merged.append(ds_m)
                
            ds_swh.close(); ds_mwd.close()
            print(f"    [{year}] OK", end='\r')
        except Exception as e: 
            print(f"Error {year}: {e}")

    print("\nMenggabungkan Dataset...")
    ds_all = xr.concat(monthly_ds_merged, dim='time')

    # Aggregates
    ds_clim = ds_all.groupby('time.month').mean()
    ds_seas = ds_all.groupby('time.season').mean()

    ranges = {
        'ts': [float(ds_all.swh.min()), float(ds_all.swh.max())],
        'clim': [float(ds_clim.swh.min()), float(ds_clim.swh.max())],
        'seas': [float(ds_seas.swh.min()), float(ds_seas.swh.max())]
    }
    
    # ==========================================
    # 3. FUNGSI PLOT
    # ==========================================
    
    def make_colorbar_img(vmin, vmax, path, label="Significant Wave Height (m)"):
        fig = plt.figure(figsize=(1.5, 6), dpi=150)
        fig.patch.set_alpha(0)
        ax = fig.add_axes([0.2, 0.05, 0.4, 0.9])
        norm = plt.Normalize(vmin=vmin, vmax=vmax)
        cb = plt.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=CMAP_MAP), cax=ax)
        cb.set_label(label, color=TEXT_COLOR_CB, weight='bold', fontsize=24)
        cb.ax.yaxis.set_tick_params(color=TEXT_COLOR_CB)
        plt.setp(cb.ax.get_yticklabels(), color=TEXT_COLOR_CB, weight='bold', fontsize=22)
        cb.outline.set_edgecolor(TEXT_COLOR_CB)
        plt.savefig(path, transparent=True, bbox_inches='tight')
        plt.close(fig)

    def plot_map_accurate(ds_slice, filename, vmin, vmax):
        lat, lon = ds_slice.latitude.values, ds_slice.longitude.values
        new_lat = np.linspace(lat.min(), lat.max(), len(lat)*4)
        new_lon = np.linspace(lon.min(), lon.max(), len(lon)*4)
        ds_interp = ds_slice.interp(latitude=new_lat, longitude=new_lon, method='linear')
        
        lon_grid, lat_grid = np.meshgrid(new_lon, new_lat)
        swh_masked = np.where(globe.is_land(lat_grid, lon_grid), np.nan, ds_interp.swh.values)

        aspect = (lon.max()-lon.min())/(lat.max()-lat.min())
        fig = plt.figure(figsize=(5*aspect, 5), dpi=DPI_MAP, frameon=False)
        ax = plt.axes(projection=ccrs.PlateCarree())
        ax.set_extent([lon.min(), lon.max(), lat.min(), lat.max()], crs=ccrs.PlateCarree())
        
        ax.pcolormesh(new_lon, new_lat, swh_masked, transform=ccrs.PlateCarree(),
                      vmin=vmin, vmax=vmax, cmap=CMAP_MAP, shading='auto', zorder=1)
        ax.add_feature(cfeature.COASTLINE, edgecolor=COASTLINE_COLOR, linewidth=1.5, zorder=2)
        
        skip = 2
        u_v, v_v = ds_slice.u_wave.values.copy(), ds_slice.v_wave.values.copy()
        l_v, ln_v = np.meshgrid(lat, lon)
        u_v[globe.is_land(l_v.T, ln_v.T)] = np.nan
        
        ax.quiver(lon[::skip], lat[::skip], u_v[::skip,::skip], v_v[::skip,::skip],
                  transform=ccrs.PlateCarree(), color=ARROW_COLOR, 
                  scale=None, scale_units='xy', alpha=0.9, width=0.005, headwidth=3.5, zorder=3)
        ax.set_axis_off()
        plt.savefig(filename, transparent=True, bbox_inches='tight', pad_inches=0)
        plt.close(fig)

    def plot_wave_rose_white(u_arr, v_arr, swh_arr, filename):
        u_f, v_f, s_f = u_arr.flatten(), v_arr.flatten(), swh_arr.flatten()
        mask = ~np.isnan(u_f) & ~np.isnan(v_f) & ~np.isnan(s_f)
        u_c, v_c, ws = u_f[mask], v_f[mask], s_f[mask]
        if len(u_c) == 0: return
        wd = np.degrees(np.arctan2(u_c, v_c)) % 360
        
        n_sectors = 16
        max_val = np.max(ws) if len(ws) > 0 else 0
        ws_bins = [0, 0.5, 1.0, 1.5, 2.0, 3.0, max_val if max_val > 3.0 else 3.1]
        
        sector_indices = (np.round(wd / (360/n_sectors)) % n_sectors).astype(int)
        ws_indices = np.digitize(ws, ws_bins) - 1
        freqs = np.zeros((len(ws_bins)-1, n_sectors))
        for i in range(len(ws)):
            if 0 <= ws_indices[i] < len(ws_bins)-1: freqs[ws_indices[i], sector_indices[i]] += 1
        freqs_pct = (freqs / len(ws)) * 100

        fig = plt.figure(figsize=(6, 6), dpi=DPI_ROSE)
        fig.patch.set_alpha(0) 
        ax = fig.add_subplot(111, polar=True)
        ax.set_facecolor('none') 
        ax.set_theta_zero_location('N')
        ax.set_theta_direction(-1)
        
        colors = plt.get_cmap(CMAP_MAP)(np.linspace(0.3, 1, len(ws_bins)-1))
        angles = np.linspace(0, 2 * np.pi, n_sectors, endpoint=False)
        width = (2 * np.pi / n_sectors) * 0.9
        bottom = 0
        labels_legend = [f"{ws_bins[i]}-{ws_bins[i+1]} m" for i in range(len(ws_bins)-1)]

        for i in range(len(ws_bins)-1):
            ax.bar(angles, freqs_pct[i, :], width=width, bottom=bottom, color=colors[i], 
                   label=labels_legend[i], alpha=0.9, edgecolor='white', linewidth=0.3)
            bottom += freqs_pct[i, :]
            
        ax.tick_params(axis='x', colors=TEXT_COLOR_ROSE, labelsize=22, pad=10)
        ax.tick_params(axis='y', colors='white', labelsize=18)
        ax.yaxis.grid(True, color='white', linestyle=':', alpha=0.4)
        ax.xaxis.grid(True, color='white', linestyle=':', alpha=0.4)
        ax.spines['polar'].set_color('white')
        ax.set_xticklabels(['N','NE','E','SE','S','SW','W','NW'])
        legend = ax.legend(loc='center left', bbox_to_anchor=(1.15, 0.5),   
                           title="SWH (m)", frameon=False,
                           fontsize=18, title_fontsize=20, labelspacing=0.8)
        plt.setp(legend.get_title(), color=TEXT_COLOR_ROSE, fontweight='bold')
        for text in legend.get_texts(): plt.setp(text, color=TEXT_COLOR_ROSE)
        plt.savefig(filename, transparent=True, bbox_inches='tight')
        plt.close(fig)

    # ==========================================
    # 4. EKSEKUSI (AREA & COLORBARS)
    # ==========================================
    print("\n Rendering Colorbars & Maps...")
    
    # Logic: Generate 1 Colorbar per mode (Sesuai Logika Angin)
    cb_paths = {
        "ts": f"{dirs['map_ts']}/colorbar_swh_ts.png",
        "clim": f"{dirs['map_clim']}/colorbar_swh_clim.png",
        "seas": f"{dirs['map_seas']}/colorbar_swh_seas.png"
    }

    make_colorbar_img(ranges['ts'][0], ranges['ts'][1], cb_paths["ts"], "Wave Height (m) [Time Series]")
    make_colorbar_img(ranges['clim'][0], ranges['clim'][1], cb_paths["clim"], "Wave Height (m) [Monthly Clim]")
    make_colorbar_img(ranges['seas'][0], ranges['seas'][1], cb_paths["seas"], "Wave Height (m) [Seasonal]")

    labels_ts = []
    for i, t in enumerate(ds_all.time):
        t_str = pd.to_datetime(t.values).strftime('%Y-%m')
        labels_ts.append(t_str)
        plot_map_accurate(ds_all.isel(time=i), f"{dirs['map_ts']}/map_{i}.png", ranges['ts'][0], ranges['ts'][1])
        plot_wave_rose_white(ds_all.u_wave.isel(time=i).values, ds_all.v_wave.isel(time=i).values, ds_all.swh.isel(time=i).values, f"{dirs['rose_area_ts']}/rose_{i}.png")
        if i % 50 == 0: print(f"    Frame TS {i}...", end='\r')

    for i in range(12):
        plot_map_accurate(ds_clim.sel(month=i+1), f"{dirs['map_clim']}/map_{i}.png", ranges['clim'][0], ranges['clim'][1])
        plot_wave_rose_white(ds_clim.u_wave.sel(month=i+1).values, ds_clim.v_wave.sel(month=i+1).values, ds_clim.swh.sel(month=i+1).values, f"{dirs['rose_area_clim']}/rose_{i}.png")

    season_order = ['DJF', 'MAM', 'JJA', 'SON']
    for i, s in enumerate(season_order):
        try:
            plot_map_accurate(ds_seas.sel(season=s), f"{dirs['map_seas']}/map_{i}.png", ranges['seas'][0], ranges['seas'][1])
            plot_wave_rose_white(ds_seas.u_wave.sel(season=s).values, ds_seas.v_wave.sel(season=s).values, ds_seas.swh.sel(season=s).values, f"{dirs['rose_area_seas']}/rose_{i}.png")
        except: pass

    # ==========================================
    # 5. GRID ROSES
    # ==========================================
    print("\n Rendering Dynamic Grid Roses...")
    lats, lons = ds_all.latitude.values, ds_all.longitude.values
    ds_u_sm = ds_all.u_wave.rolling(latitude=3, longitude=3, center=True, min_periods=1).mean()
    ds_v_sm = ds_all.v_wave.rolling(latitude=3, longitude=3, center=True, min_periods=1).mean()
    ds_s_sm = ds_all.swh.rolling(latitude=3, longitude=3, center=True, min_periods=1).mean()

    count = 0
    total_grid = len(lats) * len(lons)
    for i, lat in enumerate(lats):
        for j, lon in enumerate(lons):
            if np.isnan(ds_u_sm.values[0, i, j]): continue
            plot_wave_rose_white(ds_u_sm.values[:,i,j], ds_v_sm.values[:,i,j], ds_s_sm.values[:,i,j], f"{dirs['rose_grid_ts']}/rose_{i}_{j}.png")
            for m in range(12):
                u_m = ds_u_sm.sel(time=ds_u_sm.time.dt.month == (m+1)).values[:, i, j]
                v_m = ds_v_sm.sel(time=ds_v_sm.time.dt.month == (m+1)).values[:, i, j]
                s_m = ds_s_sm.sel(time=ds_s_sm.time.dt.month == (m+1)).values[:, i, j]
                plot_wave_rose_white(u_m, v_m, s_m, f"{dirs['rose_grid_clim']}/rose_{i}_{j}_{m}.png")
            for s_idx, s_name in enumerate(season_order):
                try:
                    u_s = ds_u_sm.sel(time=ds_u_sm.time.dt.season == s_name).values[:, i, j]
                    v_s = ds_v_sm.sel(time=ds_v_sm.time.dt.season == s_name).values[:, i, j]
                    s_s = ds_s_sm.sel(time=ds_s_sm.time.dt.season == s_name).values[:, i, j]
                    plot_wave_rose_white(u_s, v_s, s_s, f"{dirs['rose_grid_seas']}/rose_{i}_{j}_{s_idx}.png")
                except: pass
            count += 1
            if count % 20 == 0: print(f"    Grid {count}/{total_grid}...", end='\r')

    # ==========================================
    # 6. EXPORT JSON
    # ==========================================
    def to_list(da): return [round(float(x), 2) if not np.isnan(x) else None for x in da.values]
    output_data = {
        "metadata": {
            "latitudes": lats.tolist(), "longitudes": lons.tolist(),
            "labels_ts": labels_ts, "labels_clim": ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"], "labels_seas": season_order,
            "paths": {
                "map": {"ts": f"{dirs['map_ts']}/map_", "clim": f"{dirs['map_clim']}/map_", "seas": f"{dirs['map_seas']}/map_"},
                "rose": {
                    "area_ts": f"{dirs['rose_area_ts']}/rose_", "area_clim": f"{dirs['rose_area_clim']}/rose_", "area_seas": f"{dirs['rose_area_seas']}/rose_",
                    "grid_ts": f"{dirs['rose_grid_ts']}/rose_", "grid_clim": f"{dirs['rose_grid_clim']}/rose_", "grid_seas": f"{dirs['rose_grid_seas']}/rose_"
                }
            },
            "colorbar": cb_paths # Mengarah langsung ke file per mode
        },
        "area_average": {
            "ts": to_list(ds_all.swh.mean(dim=['latitude','longitude'])),
            "clim": to_list(ds_clim.swh.mean(dim=['latitude','longitude'])),
            "seas": [round(float(ds_seas.swh.sel(season=s).mean()), 2) for s in season_order]
        },
        "grid_data": {}
    }

    for i, lat in enumerate(lats):
        for j, lon in enumerate(lons):
            key = f"{i},{j}"
            if np.isnan(ds_u_sm.values[0, i, j]): continue
            output_data["grid_data"][key] = {
                "ts": to_list(ds_s_sm[:, i, j]),
                "clim": to_list(ds_clim.swh[:, i, j]),
                "seas": [round(float(ds_seas.swh.sel(season=s)[i,j]), 2) for s in season_order]
            }

    with open('wave_data_swh_rose.json', 'w') as f: json.dump(output_data, f)
    print("\n✅ SELESAI! Data SWH & Wave Rose (KE) Berhasil Di-generate.")

if __name__ == "__main__":
    process_swh_wave_rose_final()

GENERATOR V5: SWH SPATIAL + WAVE ROSES (DYNAMIC)
Membaca Data SWH & MWD 1979-2024...
    [2024] OK
Menggabungkan Dataset...

 Rendering Colorbars & Maps...


In [1]:
import xarray as xr
import numpy as np
import json
import pandas as pd
import os
import glob
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from global_land_mask import globe 
import warnings

# Matikan warning agar output bersih
warnings.filterwarnings("ignore")

def process_swh_wave_rose_final():
    print("--- 🌊 GENERATOR V5.2: SWH SPATIAL + WAVE ROSES (CLEAN JSON) ---")

    # ==========================================
    # 0. KONFIGURASI VISUAL (SINKRON DENGAN ANGIN)
    # ==========================================
    CMAP_MAP = "YlGnBu"     
    ARROW_COLOR = "#000000"  # Vektor Hitam per request
    COASTLINE_COLOR = "#000000" 
    TEXT_COLOR_ROSE = "white" # Wave Rose menggunakan tema putih kontras
    TEXT_COLOR_CB = "#e0e0e0" # Colorbar menggunakan tema abu-abu terang
    
    DPI_MAP = 150
    DPI_ROSE = 100 
    
    # 1. KONFIGURASI DATA
    start_year = 1979
    end_year = 2024
    data_folder = '.' 
    
    # Folder Output
    base_dir = "frames_swh_final"
    dirs = {
        "map_ts": f"{base_dir}/map/ts",
        "map_clim": f"{base_dir}/map/clim",
        "map_seas": f"{base_dir}/map/seas",
        "rose_area_ts": f"{base_dir}/rose/area/ts",
        "rose_area_clim": f"{base_dir}/rose/area/clim",
        "rose_area_seas": f"{base_dir}/rose/area/seas",
        "rose_grid_ts": f"{base_dir}/rose/grid/ts",
        "rose_grid_clim": f"{base_dir}/rose/grid/clim",
        "rose_grid_seas": f"{base_dir}/rose/grid/seas"
    }
    for d in dirs.values():
        os.makedirs(d, exist_ok=True)

    monthly_ds_merged = []

    # ==========================================
    # 2. LOAD DATA (SWH & MWD)
    # ==========================================
    print(f"Membaca Data SWH & MWD {start_year}-{end_year}...")

    for year in range(start_year, end_year + 1):
        swh_files = glob.glob(os.path.join(data_folder, f"*significant_height*{year}*.nc"))
        mwd_files = glob.glob(os.path.join(data_folder, f"*mean_wave_direction*{year}*.nc"))
        
        if not swh_files or not mwd_files: continue
        
        try:
            ds_swh = xr.open_dataset(swh_files[0])
            ds_mwd = xr.open_dataset(mwd_files[0])
            
            if 'valid_time' in ds_swh.coords: ds_swh = ds_swh.rename({'valid_time': 'time'})
            if 'valid_time' in ds_mwd.coords: ds_mwd = ds_mwd.rename({'valid_time': 'time'})
            
            var_swh = next((v for v in ['swh', 'significant_height'] if v in ds_swh), None)
            var_mwd = next((v for v in ['mwd', 'mean_wave_direction'] if v in ds_mwd), None)
            
            if var_swh and var_mwd:
                # Hitung komponen vektor untuk averaging (MWD ERA5 adalah "Direction To")
                u_comp = np.sin(np.deg2rad(ds_mwd[var_mwd]))
                v_comp = np.cos(np.deg2rad(ds_mwd[var_mwd]))
                
                ds_m = xr.Dataset({
                    'swh': ds_swh[var_swh].resample(time='1M').mean(),
                    'u_wave': u_comp.resample(time='1M').mean(),
                    'v_wave': v_comp.resample(time='1M').mean()
                })
                monthly_ds_merged.append(ds_m)
                
            ds_swh.close(); ds_mwd.close()
            print(f"    [{year}] OK", end='\r')
        except Exception as e: 
            print(f"Error {year}: {e}")

    print("\nMenggabungkan Dataset...")
    ds_all = xr.concat(monthly_ds_merged, dim='time')

    # Aggregates
    ds_clim = ds_all.groupby('time.month').mean()
    ds_seas = ds_all.groupby('time.season').mean()
    season_order = ['DJF', 'MAM', 'JJA', 'SON']

    ranges = {
        'ts': [float(ds_all.swh.min()), float(ds_all.swh.max())],
        'clim': [float(ds_clim.swh.min()), float(ds_clim.swh.max())],
        'seas': [float(ds_seas.swh.min()), float(ds_seas.swh.max())]
    }
    
    # ==========================================
    # 3. FUNGSI PLOT
    # ==========================================
    
    def make_colorbar_img(vmin, vmax, path, label="Significant Wave Height (m)"):
        fig = plt.figure(figsize=(1.5, 6), dpi=150)
        fig.patch.set_alpha(0)
        ax = fig.add_axes([0.2, 0.05, 0.4, 0.9])
        norm = plt.Normalize(vmin=vmin, vmax=vmax)
        cb = plt.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=CMAP_MAP), cax=ax)
        cb.set_label(label, color=TEXT_COLOR_CB, weight='bold', fontsize=24)
        cb.ax.yaxis.set_tick_params(color=TEXT_COLOR_CB)
        plt.setp(cb.ax.get_yticklabels(), color=TEXT_COLOR_CB, weight='bold', fontsize=22)
        cb.outline.set_edgecolor(TEXT_COLOR_CB)
        plt.savefig(path, transparent=True, bbox_inches='tight')
        plt.close(fig)

    def plot_map_accurate(ds_slice, filename, vmin, vmax):
        lat, lon = ds_slice.latitude.values, ds_slice.longitude.values
        new_lat = np.linspace(lat.min(), lat.max(), len(lat)*4)
        new_lon = np.linspace(lon.min(), lon.max(), len(lon)*4)
        ds_interp = ds_slice.interp(latitude=new_lat, longitude=new_lon, method='linear')
        
        lon_grid, lat_grid = np.meshgrid(new_lon, new_lat)
        swh_masked = np.where(globe.is_land(lat_grid, lon_grid), np.nan, ds_interp.swh.values)

        aspect = (lon.max()-lon.min())/(lat.max()-lat.min())
        fig = plt.figure(figsize=(5*aspect, 5), dpi=DPI_MAP, frameon=False)
        ax = plt.axes(projection=ccrs.PlateCarree())
        ax.set_extent([lon.min(), lon.max(), lat.min(), lat.max()], crs=ccrs.PlateCarree())
        
        ax.pcolormesh(new_lon, new_lat, swh_masked, transform=ccrs.PlateCarree(),
                      vmin=vmin, vmax=vmax, cmap=CMAP_MAP, shading='auto', zorder=1)
        ax.add_feature(cfeature.COASTLINE, edgecolor=COASTLINE_COLOR, linewidth=1.5, zorder=2)
        
        skip = 2
        u_v, v_v = ds_slice.u_wave.values.copy(), ds_slice.v_wave.values.copy()
        l_v, ln_v = np.meshgrid(lat, lon)
        u_v[globe.is_land(l_v.T, ln_v.T)] = np.nan
        
        ax.quiver(lon[::skip], lat[::skip], u_v[::skip,::skip], v_v[::skip,::skip],
                  transform=ccrs.PlateCarree(), color=ARROW_COLOR, 
                  scale=None, scale_units='xy', alpha=0.9, width=0.005, headwidth=3.5, zorder=3)
        ax.set_axis_off()
        plt.savefig(filename, transparent=True, bbox_inches='tight', pad_inches=0)
        plt.close(fig)

    def plot_wave_rose_white(u_arr, v_arr, swh_arr, filename):
        u_f, v_f, s_f = u_arr.flatten(), v_arr.flatten(), swh_arr.flatten()
        mask = ~np.isnan(u_f) & ~np.isnan(v_f) & ~np.isnan(s_f)
        u_c, v_c, ws = u_f[mask], v_f[mask], s_f[mask]
        if len(u_c) == 0: return
        wd = np.degrees(np.arctan2(u_c, v_c)) % 360
        
        n_sectors = 16
        max_val = np.max(ws) if len(ws) > 0 else 0
        ws_bins = [0, 0.5, 1.0, 1.5, 2.0, 3.0, max_val if max_val > 3.0 else 3.1]
        
        sector_indices = (np.round(wd / (360/n_sectors)) % n_sectors).astype(int)
        ws_indices = np.digitize(ws, ws_bins) - 1
        freqs = np.zeros((len(ws_bins)-1, n_sectors))
        for i in range(len(ws)):
            if 0 <= ws_indices[i] < len(ws_bins)-1: freqs[ws_indices[i], sector_indices[i]] += 1
        freqs_pct = (freqs / len(ws)) * 100

        fig = plt.figure(figsize=(6, 6), dpi=DPI_ROSE)
        fig.patch.set_alpha(0) 
        ax = fig.add_subplot(111, polar=True)
        ax.set_facecolor('none') 
        ax.set_theta_zero_location('N')
        ax.set_theta_direction(-1)
        
        colors = plt.get_cmap(CMAP_MAP)(np.linspace(0.3, 1, len(ws_bins)-1))
        angles = np.linspace(0, 2 * np.pi, n_sectors, endpoint=False)
        width = (2 * np.pi / n_sectors) * 0.9
        bottom = 0
        labels_legend = [f"{ws_bins[i]}-{ws_bins[i+1]} m" for i in range(len(ws_bins)-1)]

        for i in range(len(ws_bins)-1):
            ax.bar(angles, freqs_pct[i, :], width=width, bottom=bottom, color=colors[i], 
                   label=labels_legend[i], alpha=0.9, edgecolor='white', linewidth=0.3)
            bottom += freqs_pct[i, :]
            
        ax.tick_params(axis='x', colors=TEXT_COLOR_ROSE, labelsize=22, pad=10)
        ax.tick_params(axis='y', colors='white', labelsize=18)
        ax.yaxis.grid(True, color='white', linestyle=':', alpha=0.4)
        ax.xaxis.grid(True, color='white', linestyle=':', alpha=0.4)
        ax.spines['polar'].set_color('white')
        ax.set_xticklabels(['N','NE','E','SE','S','SW','W','NW'])
        legend = ax.legend(loc='center left', bbox_to_anchor=(1.15, 0.5),   
                           title="SWH (m)", frameon=False,
                           fontsize=18, title_fontsize=20, labelspacing=0.8)
        plt.setp(legend.get_title(), color=TEXT_COLOR_ROSE, fontweight='bold')
        for text in legend.get_texts(): plt.setp(text, color=TEXT_COLOR_ROSE)
        plt.savefig(filename, transparent=True, bbox_inches='tight')
        plt.close(fig)

    # ==========================================
    # 4. EKSEKUSI (AREA & COLORBARS)
    # ==========================================
    print("\n Rendering Colorbars & Maps...")
    cb_paths = {
        "ts": f"{dirs['map_ts']}/colorbar_swh_ts.png",
        "clim": f"{dirs['map_clim']}/colorbar_swh_clim.png",
        "seas": f"{dirs['map_seas']}/colorbar_swh_seas.png"
    }
    make_colorbar_img(ranges['ts'][0], ranges['ts'][1], cb_paths["ts"], "Wave Height (m) [Time Series]")
    make_colorbar_img(ranges['clim'][0], ranges['clim'][1], cb_paths["clim"], "Wave Height (m) [Monthly Clim]")
    make_colorbar_img(ranges['seas'][0], ranges['seas'][1], cb_paths["seas"], "Wave Height (m) [Seasonal]")

    labels_ts = []
    for i, t in enumerate(ds_all.time):
        t_str = pd.to_datetime(t.values).strftime('%Y-%m')
        labels_ts.append(t_str)
        plot_map_accurate(ds_all.isel(time=i), f"{dirs['map_ts']}/map_{i}.png", ranges['ts'][0], ranges['ts'][1])
        plot_wave_rose_white(ds_all.u_wave.isel(time=i).values, ds_all.v_wave.isel(time=i).values, ds_all.swh.isel(time=i).values, f"{dirs['rose_area_ts']}/rose_{i}.png")
        if i % 50 == 0: print(f"    Frame TS {i}...", end='\r')

    for i in range(12):
        plot_map_accurate(ds_clim.sel(month=i+1), f"{dirs['map_clim']}/map_{i}.png", ranges['clim'][0], ranges['clim'][1])
        plot_wave_rose_white(ds_clim.u_wave.sel(month=i+1).values, ds_clim.v_wave.sel(month=i+1).values, ds_clim.swh.sel(month=i+1).values, f"{dirs['rose_area_clim']}/rose_{i}.png")

    for i, s in enumerate(season_order):
        try:
            plot_map_accurate(ds_seas.sel(season=s), f"{dirs['map_seas']}/map_{i}.png", ranges['seas'][0], ranges['seas'][1])
            plot_wave_rose_white(ds_seas.u_wave.sel(season=s).values, ds_seas.v_wave.sel(season=s).values, ds_seas.swh.sel(season=s).values, f"{dirs['rose_area_seas']}/rose_{i}.png")
        except: pass

    # ==========================================
    # 5. GRID ROSES
    # ==========================================
    print("\n Rendering Dynamic Grid Roses...")
    lats, lons = ds_all.latitude.values, ds_all.longitude.values
    ds_u_sm = ds_all.u_wave.rolling(latitude=3, longitude=3, center=True, min_periods=1).mean()
    ds_v_sm = ds_all.v_wave.rolling(latitude=3, longitude=3, center=True, min_periods=1).mean()
    ds_s_sm = ds_all.swh.rolling(latitude=3, longitude=3, center=True, min_periods=1).mean()

    count = 0
    total_grid = len(lats) * len(lons)
    for i, lat in enumerate(lats):
        for j, lon in enumerate(lons):
            if np.isnan(ds_u_sm.values[0, i, j]): continue
            plot_wave_rose_white(ds_u_sm.values[:,i,j], ds_v_sm.values[:,i,j], ds_s_sm.values[:,i,j], f"{dirs['rose_grid_ts']}/rose_{i}_{j}.png")
            for m in range(12):
                u_m = ds_u_sm.sel(time=ds_u_sm.time.dt.month == (m+1)).values[:, i, j]
                v_m = ds_v_sm.sel(time=ds_v_sm.time.dt.month == (m+1)).values[:, i, j]
                s_m = ds_s_sm.sel(time=ds_s_sm.time.dt.month == (m+1)).values[:, i, j]
                plot_wave_rose_white(u_m, v_m, s_m, f"{dirs['rose_grid_clim']}/rose_{i}_{j}_{m}.png")
            for s_idx, s_name in enumerate(season_order):
                try:
                    u_s = ds_u_sm.sel(time=ds_u_sm.time.dt.season == s_name).values[:, i, j]
                    v_s = ds_v_sm.sel(time=ds_v_sm.time.dt.season == s_name).values[:, i, j]
                    s_s = ds_s_sm.sel(time=ds_s_sm.time.dt.season == s_name).values[:, i, j]
                    plot_wave_rose_white(u_s, v_s, s_s, f"{dirs['rose_grid_seas']}/rose_{i}_{j}_{s_idx}.png")
                except: pass
            count += 1
            if count % 20 == 0: print(f"    Grid {count}/{total_grid}...", end='\r')

    # ==========================================
    # 6. EXPORT JSON (SUNTIKAN LOGIKA TERBARU: ANTI-NaN & METADATA LENGKAP)
    # ==========================================
    
    # Fungsi pembantu untuk membersihkan nilai NaN sebelum masuk JSON
    def clean_val(x):
        try:
            val = float(x)
            return round(val, 2) if np.isfinite(val) else None
        except:
            return None

    def to_list_clean(da):
        return [clean_val(x) for x in da.values]

    output_data = {
        "metadata": {
            "latitudes": lats.tolist(), "longitudes": lons.tolist(),
            "labels_ts": labels_ts, 
            "labels_clim": ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"], 
            "labels_seas": season_order,
            "paths": {
                "map": {"ts": f"{dirs['map_ts']}/map_", "clim": f"{dirs['map_clim']}/map_", "seas": f"{dirs['map_seas']}/map_"},
                "rose": {
                    "area_ts": f"{dirs['rose_area_ts']}/rose_", "area_clim": f"{dirs['rose_area_clim']}/rose_", "area_seas": f"{dirs['rose_area_seas']}/rose_",
                    "grid_ts": f"{dirs['rose_grid_ts']}/rose_", "grid_clim": f"{dirs['rose_grid_clim']}/rose_", "grid_seas": f"{dirs['rose_grid_seas']}/rose_"
                }
            },
            "colorbar": cb_paths,
            "bounds": [[float(lats.min()), float(lons.min())], [float(lats.max()), float(lons.max())]],
            "color_ranges": ranges
        },
        "area_average": {
            "ts": to_list_clean(ds_all.swh.mean(dim=['latitude','longitude'])),
            "clim": to_list_clean(ds_clim.swh.mean(dim=['latitude','longitude'])),
            "seas": [clean_val(ds_seas.swh.sel(season=s).mean()) for s in season_order]
        },
        "grid_data": {}
    }

    for i in range(len(lats)):
        for j in range(len(lons)):
            key = f"{i},{j}"
            if np.isnan(ds_u_sm.values[:, i, j]).all(): continue
            output_data["grid_data"][key] = {
                "ts": to_list_clean(ds_s_sm[:, i, j]),
                "clim": to_list_clean(ds_clim.swh[:, i, j]),
                "seas": [clean_val(ds_seas.swh.sel(season=s).values[i,j]) for s in season_order]
            }

    with open('wave_data_swh_rose.json', 'w') as f: 
        json.dump(output_data, f)
        
    print("\n✅ BERHASIL! File 'wave_data_swh_rose.json' telah di-generate tanpa nilai NaN.")

if __name__ == "__main__":
    process_swh_wave_rose_final()

--- 🌊 GENERATOR V5.2: SWH SPATIAL + WAVE ROSES (CLEAN JSON) ---
Membaca Data SWH & MWD 1979-2024...
    [2024] OK
Menggabungkan Dataset...

 Rendering Colorbars & Maps...
    Frame TS 550...
 Rendering Dynamic Grid Roses...
    Grid 280/289...
✅ BERHASIL! File 'wave_data_swh_rose.json' telah di-generate tanpa nilai NaN.
